Cell 1 — Install dependencies + connect Groq API + SQLite DB

In [44]:
!pip install -q langchain langchain-community langchain-groq langgraph pypdf

In [45]:
import os, sqlite3, logging
from langchain_community.utilities import SQLDatabase
from datetime import datetime
from pathlib import Path

# Use Groq key stored in Colab > Secrets > "GROQ_API_KEY"
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "")

# Download example DB
!wget -q [github.com](https://github.com/jayyanar/agentic-ai-training/raw/lab-day-1/batch2/lca-langchainV1-essentials/Chinook.db)
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
print(" Connected DB tables:", db.get_usable_table_names())

# Logging folder
LOG_DIR = Path("./logs"); LOG_DIR.mkdir(exist_ok=True)
logging.basicConfig(filename=LOG_DIR / f"log_{datetime.now().strftime('%H%M%S')}.log",
                    level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")


/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `wget -q [github.com](https://github.com/jayyanar/agentic-ai-training/raw/lab-day-1/batch2/lca-langchainV1-essentials/Chinook.db)'
 Connected DB tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track', 'document_log']


Document Loader Agent

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
from pathlib import Path

DATA_DIR = Path("/content/data/pdfs")
DATA_DIR.mkdir(parents=True, exist_ok=True)

for file_name in uploaded.keys():
    os.rename(file_name, DATA_DIR / file_name)

print(" Files moved to:", DATA_DIR)

In [46]:
from langchain_community.document_loaders import PyPDFLoader
import glob

class DocumentLoaderAgent:
    """Loads all PDFs from provided folder and extracts text."""
    def __init__(self, folder: Path): self.folder = folder
    def load_all(self):
        pdfs = sorted(glob.glob(str(self.folder / "*.pdf")))
        docs=[]
        print(f" Found {len(pdfs)} PDFs")
        for p in pdfs:
            try:
                text=" ".join([pg.page_content for pg in PyPDFLoader(p).load_and_split()])
                docs.append({
                    "document_name": Path(p).name,
                    "received_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "text": text})
                print(f" Loaded {Path(p).name}")
            except Exception as e: print(f" {Path(p).name} -> {e}")
        return docs

DATA_DIR = Path("/content/data/pdfs")
Path(DATA_DIR).mkdir(parents=True,exist_ok=True)
loader_agent = DocumentLoaderAgent(DATA_DIR)
documents = loader_agent.load_all()


 Found 29 PDFs
 Loaded 01_copyright_infringement_photography.pdf
 Loaded 02_trademark_infringement_tech.pdf
 Loaded 03_trade_secret_misappropriation.pdf
 Loaded 04_defamation_online_review.pdf
 Loaded 05_patent_infringement_medical_device.pdf
 Loaded 06_harassment_workplace.pdf
 Loaded 07_software_license_violation.pdf
 Loaded 08_non_compete_violation.pdf
 Loaded 09_copyright_infringement_music.pdf
 Loaded 10_breach_of_contract_nda.pdf
 Loaded LOA2.pdf
 Loaded LOA3.pdf
 Loaded LOA4.pdf
 Loaded LOA5.pdf
 Loaded LOA6.pdf
 Loaded LOA7.pdf
 Loaded LOA8.pdf
 Loaded LOA9.pdf
 Loaded LoA1.pdf
 Loaded bw_doc_1.pdf
 Loaded bw_doc_2.pdf
 Loaded bw_doc_3.pdf
 Loaded bw_doc_4.pdf
 Loaded bw_doc_5.pdf
 Loaded notice_1.pdf
 Loaded notice_2.pdf
 Loaded notice_3.pdf
 Loaded notice_4.pdf
 Loaded notice_5.pdf


Classification Agent

In [ ]:
import os
print(os.getenv("GROQ_API_KEY"))

In [48]:
from langchain_groq import ChatGroq
from typing import Dict, List
import json
import re
import os
import time

#  LLM (make sure API key is already set)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

#  SYSTEM PROMPT
SYSTEM_PROMPT = """
You are a strict legal document classifier.

Classify into ONE:

1. Cease
- Formal legal notice
- Explicit demand to STOP activity
- Mentions legal violations or enforcement

2. Irrelevant
- Debt / Power of Attorney / financial documents
- Administrative forms
- "Cease communication" (NOT legal enforcement)

3. Uncertain
- Empty / scanned / unreadable
- Very short or unclear

STRICT RULES:
- "Cease communication" ≠ Cease
- Must be legal enforcement to be Cease
- If unsure → Uncertain

Return ONLY JSON:
{
  "classification": "Cease/Uncertain/Irrelevant",
  "explanation": "short reason",
  "confidence": 0.X
}
"""

class ClassificationAgent:

    def __init__(self):
        self.llm = llm


    #  SINGLE DOCUMENT CLASSIFICATION

    def classify_document(self, doc: Dict) -> Dict:

        try:
            text = (doc.get("text") or "").lower()

            #  RULE 1: Empty / OCR issues
            if len(text.strip()) < 200:
                result = {
                    "classification": "Uncertain",
                    "explanation": "Text too short / unreadable",
                    "confidence": 0.0
                }

            #  RULE 2: Debt / POA
            elif any(k in text for k in [
                "power of attorney", "debt", "creditor",
                "account", "settlement", "financial records"
            ]):
                result = {
                    "classification": "Irrelevant",
                    "explanation": "Debt/POA document",
                    "confidence": 0.95
                }


            #  LLM FALLBACK

            else:
                prompt = SYSTEM_PROMPT + f"\n\nDocument:\n{text[:1200]}"

                max_retries = 3
                data = None

                for attempt in range(max_retries):
                    try:
                        res = self.llm.invoke(prompt)
                        raw = res.content.strip()

                        json_match = re.search(r'\{.*\}', raw, re.DOTALL)

                        if json_match:
                            data = json.loads(json_match.group())
                        else:
                            raise ValueError("No JSON found")

                        break

                    except Exception as e:
                        print(f" Attempt {attempt+1} failed for {doc.get('document_name')}: {e}")
                        time.sleep(2)

                if data is None:
                    result = {
                        "classification": "Uncertain",
                        "explanation": "LLM failed",
                        "confidence": 0.0
                    }
                else:
                    result = {
                        "classification": data.get("classification", "Uncertain"),
                        "explanation": data.get("explanation", "LLM output"),
                        "confidence": data.get("confidence", 0.5)
                    }


            #  NORMALIZE OUTPUT

            if result["classification"] not in ["Cease", "Uncertain", "Irrelevant"]:
                result["classification"] = "Uncertain"

            if not isinstance(result.get("confidence"), (int, float)):
                result["confidence"] = 0.0

            #  Add metadata
            result.update({
                "document_name": doc.get("document_name"),
                "received_date": doc.get("received_date")
            })

            print(f" {result['document_name']} → {result['classification']} ({result['confidence']})")

            return result

        except Exception as e:
            print(f" CRITICAL ERROR in document: {doc.get('document_name')} → {e}")

            return {
                "classification": "Uncertain",
                "explanation": f"Critical error: {e}",
                "confidence": 0.0,
                "document_name": doc.get("document_name"),
                "received_date": doc.get("received_date")
            }


    #  BATCH PROCESSING (SAFE LOOP)

    def classify_batch(self, docs: List[Dict]) -> List[Dict]:

        results = []

        print(f"\n Processing {len(docs)} documents...\n")

        for i, d in enumerate(docs):
            try:
                print(f" {i+1}/{len(docs)}: {d.get('document_name')}")

                #  Safety check
                if "text" not in d or not isinstance(d["text"], str):
                    print(" Invalid text → marking Uncertain")

                    results.append({
                        "classification": "Uncertain",
                        "explanation": "Invalid or missing text",
                        "confidence": 0.0,
                        "document_name": d.get("document_name"),
                        "received_date": d.get("received_date")
                    })
                    continue

                result = self.classify_document(d)
                results.append(result)

                time.sleep(1)

            except Exception as e:
                print(f" ERROR on {d.get('document_name')}: {e}")

                results.append({
                    "classification": "Uncertain",
                    "explanation": f"Batch error: {e}",
                    "confidence": 0.0,
                    "document_name": d.get("document_name"),
                    "received_date": d.get("received_date")
                })

                continue

        print(f"\n Completed processing {len(results)} documents")

        return results



#  RUN

classifier = ClassificationAgent()
classified_docs = classifier.classify_batch(documents)


 Processing 29 documents...

 1/29: 01_copyright_infringement_photography.pdf
 01_copyright_infringement_photography.pdf → Irrelevant (0.95)
 2/29: 02_trademark_infringement_tech.pdf
 02_trademark_infringement_tech.pdf → Cease (1.0)
 3/29: 03_trade_secret_misappropriation.pdf
 03_trade_secret_misappropriation.pdf → Irrelevant (0.95)
 4/29: 04_defamation_online_review.pdf
 04_defamation_online_review.pdf → Cease (1.0)
 5/29: 05_patent_infringement_medical_device.pdf
 05_patent_infringement_medical_device.pdf → Irrelevant (0.95)
 6/29: 06_harassment_workplace.pdf
 06_harassment_workplace.pdf → Irrelevant (0.95)
 7/29: 07_software_license_violation.pdf
 07_software_license_violation.pdf → Cease (1.0)
 8/29: 08_non_compete_violation.pdf
 08_non_compete_violation.pdf → Irrelevant (0.95)
 9/29: 09_copyright_infringement_music.pdf
 09_copyright_infringement_music.pdf → Cease (1.0)
 10/29: 10_breach_of_contract_nda.pdf
 10_breach_of_contract_nda.pdf → Irrelevant (0.95)
 11/29: LOA2.pdf
 LOA2.

In [13]:
print("Total documents loaded:", len(documents))

Total documents loaded: 29


Database + Archiving + Audit Agents

In [20]:
!wget -q https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite

In [21]:
import os
os.rename("Chinook_Sqlite.sqlite", "Chinook.db")

In [22]:
db_agent = DatabaseAgent("Chinook.db")

In [23]:
print(db_agent.db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [49]:
from langchain_community.utilities import SQLDatabase
from datetime import datetime


#  DATABASE AGENT

class DatabaseAgent:

    def __init__(self, db_uri="Chinook.db"):
        self.db = SQLDatabase.from_uri(f"sqlite:///{db_uri}")

        #  Create proper table (FIX)
        self.db.run("""
        CREATE TABLE IF NOT EXISTS document_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            document_name TEXT,
            classification TEXT,
            received_date TEXT
        );
        """)

        print(" document_log table ready")

    def insert_record(self, doc, status):
        try:
            name = doc["document_name"].replace("'", " ")
            date = doc["received_date"]

            query = f"""
            INSERT INTO document_log (document_name, classification, received_date)
            VALUES ('{name}', '{status}', '{date}');
            """

            self.db.run(query)

            print(f" [DB] {status} recorded → {name}")

        except Exception as e:
            print(f" DB insert error ({status}):", e)



#  ARCHIVING AGENT

class ArchivingAgent:

    def __init__(self, db: DatabaseAgent):
        self.db = db

    def archive(self, doc):
        self.db.insert_record(doc, "IRRELEVANT")
        print(f" Archived → {doc['document_name']}")



#  AUDIT AGENT

class AuditAgent:

    def __init__(self):
        self.logs = []

    def record(self, action, detail):
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "action": action,
            "detail": detail
        }

        self.logs.append(entry)

        print(f" [AUDIT] {entry['timestamp']} | {action} → {detail}")



#  PROCESSING AGENT (ORCHESTRATOR)

class ProcessingAgent:

    def __init__(self, db_agent, archiver, auditor):
        self.db = db_agent
        self.archiver = archiver
        self.auditor = auditor

    def process(self, classified_docs):

        print(f"\n Processing {len(classified_docs)} classified documents...\n")

        for i, doc in enumerate(classified_docs):
            try:
                name = doc.get("document_name")
                cls = doc.get("classification")

                print(f" {i+1}/{len(classified_docs)} → {name} ({cls})")

                if cls == "Cease":
                    self.db.insert_record(doc, "CEASE")
                    self.auditor.record("CEASE_RECORDED", name)

                elif cls == "Irrelevant":
                    self.archiver.archive(doc)
                    self.auditor.record("ARCHIVED", name)

                elif cls == "Uncertain":
                    self.db.insert_record(doc, "UNCERTAIN")
                    self.auditor.record("FLAGGED_FOR_REVIEW", name)

                else:
                    self.auditor.record("UNKNOWN_CLASSIFICATION", name)

            except Exception as e:
                print(f" ERROR processing {doc.get('document_name')}: {e}")
                self.auditor.record("PROCESSING_ERROR", f"{doc.get('document_name')} → {e}")

        print("\n Completed processing all documents")



#  INITIALIZE AGENTS

db_agent = DatabaseAgent("Chinook.db")
archiver = ArchivingAgent(db_agent)
auditor = AuditAgent()

processor = ProcessingAgent(db_agent, archiver, auditor)


#  RUN PIPELINE

processor.process(classified_docs)

 document_log table ready

 Processing 29 classified documents...

 1/29 → 01_copyright_infringement_photography.pdf (Irrelevant)
 [DB] IRRELEVANT recorded → 01_copyright_infringement_photography.pdf
 Archived → 01_copyright_infringement_photography.pdf
 [AUDIT] 2026-03-23 17:02:42 | ARCHIVED → 01_copyright_infringement_photography.pdf
 2/29 → 02_trademark_infringement_tech.pdf (Cease)
 [DB] CEASE recorded → 02_trademark_infringement_tech.pdf
 [AUDIT] 2026-03-23 17:02:42 | CEASE_RECORDED → 02_trademark_infringement_tech.pdf
 3/29 → 03_trade_secret_misappropriation.pdf (Irrelevant)
 [DB] IRRELEVANT recorded → 03_trade_secret_misappropriation.pdf
 Archived → 03_trade_secret_misappropriation.pdf
 [AUDIT] 2026-03-23 17:02:42 | ARCHIVED → 03_trade_secret_misappropriation.pdf
 4/29 → 04_defamation_online_review.pdf (Cease)
 [DB] CEASE recorded → 04_defamation_online_review.pdf
 [AUDIT] 2026-03-23 17:02:42 | CEASE_RECORDED → 04_defamation_online_review.pdf
 5/29 → 05_patent_infringement_medic

Human‑in‑the‑Loop Agent (approve/reject)

In [50]:
class HITLAgent:
    """Human-in-the-loop review for low-confidence documents."""

    def review(self, doc):
        print("\n" + "="*50)
        print(" HUMAN REVIEW REQUIRED")
        print("="*50)

        print(f" Document: {doc.get('document_name')}")
        print(f" Confidence: {doc.get('confidence', 0)}")
        print(f" Explanation: {doc.get('explanation', '')[:200]}...\n")

        print(" Options:")
        print("  [A] Approve  → Cease")
        print("  [R] Reject   → Irrelevant")
        print("  [S] Skip     → Keep as Uncertain")

        while True:
            try:
                user_input = input("Enter choice (A/R/S): ").strip().lower()

                if user_input in ["a", "approve", "cease", "c"]:
                    return "Cease"

                elif user_input in ["r", "reject", "irrelevant", "i"]:
                    return "Irrelevant"

                elif user_input in ["s", "skip"]:
                    return "Uncertain"

                else:
                    print(" Invalid input. Please type A / R / S")

            except Exception as e:
                print(" Input error, defaulting to Uncertain")
                return "Uncertain"


#  Initialize
hitl = HITLAgent()

LangGraph Workflow

In [51]:
from langgraph.graph import StateGraph, START, END


#  NODE DEFINITIONS


def router_node(state):
    return state


def cease_node(state):
    doc = state["doc"]

    db_agent.insert_record(doc, "CEASE")
    auditor.record("CEASE_RECORDED", doc["document_name"])

    return {}


def irr_node(state):
    doc = state["doc"]

    db_agent.insert_record(doc, "IRRELEVANT")
    auditor.record("ARCHIVED", doc["document_name"])

    return {}


def hitl_node(state):
    doc = state["doc"]

    decision = hitl.review(doc)

    if decision == "Cease":
        db_agent.insert_record(doc, "CEASE")
        auditor.record("HITL_APPROVED", doc["document_name"])

    elif decision == "Irrelevant":
        archiver.archive(doc)
        auditor.record("HITL_REJECTED", doc["document_name"])

    else:
        db_agent.insert_record(doc, "UNCERTAIN")
        auditor.record("HITL_SKIPPED", doc["document_name"])

    return {}



#  ROUTING FUNCTION


def route(state):
    c = state["doc"]["classification"]

    if c == "Cease":
        return "cease"
    elif c == "Irrelevant":
        return "irrelevant"
    else:
        return "hitl"



#  BUILD GRAPH


flow = StateGraph(dict)

flow.add_node("router", router_node)
flow.add_node("cease", cease_node)
flow.add_node("irrelevant", irr_node)
flow.add_node("hitl", hitl_node)

flow.add_edge(START, "router")

#  IMPORTANT FIX
flow.add_conditional_edges(
    "router",
    route,
    {
        "cease": "cease",
        "irrelevant": "irrelevant",
        "hitl": "hitl"
    }
)

flow.add_edge("cease", END)
flow.add_edge("irrelevant", END)
flow.add_edge("hitl", END)

graph = flow.compile()

print(" LangGraph compiled successfully (runtime mode)")


#  EXECUTION


print("\n --- Starting Document Processing via LangGraph ---\n")

for i, doc in enumerate(classified_docs):
    try:
        print(f"\n {i+1}/{len(classified_docs)} → {doc['document_name']}")
        print(f" Classification: {doc['classification']}")

        graph.invoke({
            "doc": doc   #  THIS IS THE KEY
        })

    except Exception as e:
        print(f" ERROR processing {doc['document_name']}: {e}")

print("\n --- All documents processed through LangGraph ---")

 LangGraph compiled successfully (runtime mode)

 --- Starting Document Processing via LangGraph ---


 1/29 → 01_copyright_infringement_photography.pdf
 Classification: Irrelevant
 [DB] IRRELEVANT recorded → 01_copyright_infringement_photography.pdf
 [AUDIT] 2026-03-23 17:04:16 | ARCHIVED → 01_copyright_infringement_photography.pdf

 2/29 → 02_trademark_infringement_tech.pdf
 Classification: Cease
 [DB] CEASE recorded → 02_trademark_infringement_tech.pdf
 [AUDIT] 2026-03-23 17:04:16 | CEASE_RECORDED → 02_trademark_infringement_tech.pdf

 3/29 → 03_trade_secret_misappropriation.pdf
 Classification: Irrelevant
 [DB] IRRELEVANT recorded → 03_trade_secret_misappropriation.pdf
 [AUDIT] 2026-03-23 17:04:16 | ARCHIVED → 03_trade_secret_misappropriation.pdf

 4/29 → 04_defamation_online_review.pdf
 Classification: Cease
 [DB] CEASE recorded → 04_defamation_online_review.pdf
 [AUDIT] 2026-03-23 17:04:16 | CEASE_RECORDED → 04_defamation_online_review.pdf

 5/29 → 05_patent_infringement_medical_

Audit Log

In [52]:
from datetime import datetime
from langgraph.graph import StateGraph, START, END


#  AUDIT AGENT

class AuditAgent:

    def __init__(self):
        self.logs = []

    def record(self, action, detail):
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "action": action,
            "detail": detail
        }

        self.logs.append(entry)

        print(f" [AUDIT] {entry['timestamp']} | {action} → {detail}")


#  Initialize auditor (GLOBAL)
auditor = AuditAgent()


# NODES


def router_node(state):
    return state


def cease_node(state):
    doc = state["doc"]
    print(" CEASE NODE HIT")
    auditor.record("CEASE_RECORDED", doc["document_name"])
    return {}


def irr_node(state):
    doc = state["doc"]
    print(" IRRELEVANT NODE HIT")
    auditor.record("ARCHIVED", doc["document_name"])
    return {}



# ROUTING


def route(state):
    c = state["doc"].get("classification", "").strip()

    if c == "Cease":
        return "cease"
    else:
        return "irrelevant"



#BUILD GRAPH


flow = StateGraph(dict)

flow.add_node("router", router_node)
flow.add_node("cease", cease_node)
flow.add_node("irrelevant", irr_node)

flow.add_edge(START, "router")

flow.add_conditional_edges(
    "router",
    route,
    {
        "cease": "cease",
        "irrelevant": "irrelevant"
    }
)

flow.add_edge("cease", END)
flow.add_edge("irrelevant", END)

graph = flow.compile()

print(" Graph ready")


#  EXECUTION

#  SAMPLE DATA (replace with your classified_docs)
classified_docs = [
    {"document_name": "doc1.pdf", "classification": "Cease"},
    {"document_name": "doc2.pdf", "classification": "Irrelevant"},
    {"document_name": "doc3.pdf", "classification": "Uncertain"}
]

print("\n Running pipeline...\n")

for doc in classified_docs:
    print(f"\n Processing: {doc['document_name']} | {doc['classification']}")

    graph.invoke({
        "doc": doc
    })

print("\n Done\n")


#  PRINT AUDIT LOGS


print("\n FINAL AUDIT TRAIL:\n")

for log in auditor.logs:
    print(log)

 Graph ready

 Running pipeline...


 Processing: doc1.pdf | Cease
 CEASE NODE HIT
 [AUDIT] 2026-03-23 17:04:58 | CEASE_RECORDED → doc1.pdf

 Processing: doc2.pdf | Irrelevant
 IRRELEVANT NODE HIT
 [AUDIT] 2026-03-23 17:04:58 | ARCHIVED → doc2.pdf

 Processing: doc3.pdf | Uncertain
 IRRELEVANT NODE HIT
 [AUDIT] 2026-03-23 17:04:58 | ARCHIVED → doc3.pdf

 Done


 FINAL AUDIT TRAIL:

{'timestamp': '2026-03-23 17:04:58', 'action': 'CEASE_RECORDED', 'detail': 'doc1.pdf'}
{'timestamp': '2026-03-23 17:04:58', 'action': 'ARCHIVED', 'detail': 'doc2.pdf'}
{'timestamp': '2026-03-23 17:04:58', 'action': 'ARCHIVED', 'detail': 'doc3.pdf'}


In [56]:

# Process ALL Documents + Show FULL Audit Log


thread = {"configurable": {"thread_id": "batch‑run‑001"}}

print("\n BATCH WORKFLOW START\n" + "═" * 40)
stats = {"Cease": 0, "Irrelevant": 0, "HITL": 0}

for idx, doc in enumerate(classified_docs, start=1):
    print(f"\n--- RECEIPT OF DOCUMENT {idx}/{len(classified_docs)} ---")
    print(f"Document: {doc['document_name']} | Classification: {doc['classification']}")

    # Step 1 – invoke graph for current document
    graph.invoke({"doc": doc}, config=thread)

    # Step 2 – handle human review if uncertain
    if doc.get("classification") == "Uncertain":
        print("\n--- EXECUTION PAUSED FOR HUMAN REVIEW ---")
        print(f"Explanation: {doc.get('explanation','')[:200]} ...")

        decision = input("Approve (A = Cease) / Reject (R = Irrelevant): ").strip().lower()

        if decision.startswith("a"):
            doc["classification"] = "Cease"
        else:
            doc["classification"] = "Irrelevant"

        print(f" Human decision: {doc['classification']}")
        graph.invoke({"doc": doc}, config=thread)   # resume processing

        stats["HITL"] += 1
    else:
        stats[doc["classification"]] += 1

print("\n All documents processed.")
print(" Summary:", stats)


# Print the COMPLETE audit log for this batch

print("\n FULL AUDIT LOG\n" + "─" * 50)
if hasattr(auditor, "logs") and auditor.logs:
    for i, entry in enumerate(auditor.logs, start=1):
        # if your AuditAgent stores dict entries
        if isinstance(entry, dict):
            ts = entry.get("timestamp", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            act = entry.get("action", "")
            det = entry.get("detail", "")
            print(f"{i:02}. {ts} | {act} → {det}")
        else:
            print(f"{i:02}. {entry}")
else:
    print("No audit records found – check earlier processing steps.")

print("\n Audit trail displayed successfully.")



 BATCH WORKFLOW START
════════════════════════════════════════

--- RECEIPT OF DOCUMENT 1/3 ---
Document: doc1.pdf | Classification: Cease
 CEASE NODE HIT
 [AUDIT] 2026-03-23 17:13:22 | CEASE_RECORDED → doc1.pdf

--- RECEIPT OF DOCUMENT 2/3 ---
Document: doc2.pdf | Classification: Irrelevant
 IRRELEVANT NODE HIT
 [AUDIT] 2026-03-23 17:13:22 | ARCHIVED → doc2.pdf

--- RECEIPT OF DOCUMENT 3/3 ---
Document: doc3.pdf | Classification: Irrelevant
 IRRELEVANT NODE HIT
 [AUDIT] 2026-03-23 17:13:22 | ARCHIVED → doc3.pdf

 All documents processed.
 Summary: {'Cease': 1, 'Irrelevant': 2, 'HITL': 0}

 FULL AUDIT LOG
──────────────────────────────────────────────────
01. 2026-03-23 17:04:58 | CEASE_RECORDED → doc1.pdf
02. 2026-03-23 17:04:58 | ARCHIVED → doc2.pdf
03. 2026-03-23 17:04:58 | ARCHIVED → doc3.pdf
04. 2026-03-23 17:05:56 | CEASE_RECORDED → doc1.pdf
05. 2026-03-23 17:10:27 | CEASE_RECORDED → doc1.pdf
06. 2026-03-23 17:12:04 | CEASE_RECORDED → doc1.pdf
07. 2026-03-23 17:12:04 | ARCHIVED 